# 전체 CLV 개입 예산을 N/V에 나눈 저차원 M2 — Dunnhumby seed 42

기존 구조는 그대로 두고 N축과 V축의 고정 계수만 각각 `0.10 → 0.05`로 낮추어, 전체 CLV 개입 예산을 `0.10`으로 맞춘 한 점을 평가합니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 표현: ID 64차원 + 거래활동 4차원 + 거래당 가치 4차원
- 점수: `S = S_ID + 0.05 S_N + 0.05 S_V`
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch
- 제거 상태 유지: 사용자 게이트, 학습형 축 가중치, 명시적 아이템 인기도·가격 입력

이 실행은 역사적 개발구간의 seed 42 탐색이며 유의성을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '__PIN_AFTER_COMMIT__'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_gatefree_lowdim import (
    configure_gatefree_lowdim_run,
    preflight_summary,
    run_gatefree_lowdim_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_gatefree_lowdim_run(
    axis_budget=0.05,
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_split_budget_v2'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['m2']['fixed_per_axis_budget'] == 0.05
assert summary['m2']['fixed_total_clv_budget'] == 0.10
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_gatefree_lowdim_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)